# 📋 Odpowiedzi: Function Calling

*Wygenerowano automatycznie — nie commitować!*

---
### 1. Ćwiczenie 1: Co ważniejsze — nazwa czy opis narzędzia?

LLM wybiera narzędzie na podstawie **nazwy** (`name`) i **opisu** (`description`).
Ale co ma większy wpływ na jego decyzję? Sprawdźmy!

**Część A — Zadanie: zmień opis i sprawdź efekt**

Zmień `description` narzędzia `get_weather` na coś kompletnie mylącego.
Zadaj pytanie o pogodę i obserwuj: czy LLM się pomylił?

In [ ]:
# Ćwiczenie 1A — rozwiązanie
import copy

test_tools = copy.deepcopy(tools_definition)
test_tools[0]["function"]["description"] = "Przeszukuje bazę danych o prezydentach Polski"

---
### 2. Ćwiczenie 2: Dodaj własne narzędzie

Stwórz nową funkcję i dodaj jej opis do `tools_definition`.

**Zadanie:** Napisz narzędzie `get_population(city)` które zwraca przybliżoną liczbę
mieszkańców polskiego miasta.

**Dane do użycia** (GUS 2024):

| Miasto | Mieszkańcy | Ranking |
|---|---|---|
| Warszawa | ~1 860 000 | #1 |
| Kraków | ~800 000 | #2 |
| Wrocław | ~674 000 | #3 |
| Łódź | ~646 000 | #4 |
| Poznań | ~535 000 | #5 |
| Gdańsk | ~470 000 | #6 |

**Jak to zrobić:**
1. Przechowaj dane w **słowniku** `dict` — klucz to nazwa miasta, wartość to **tupla** `(populacja, ranking)`
2. Funkcja zwraca **f-string** z informacjami, np. `"Kraków: ~800,000 mieszkańców (#2 w Polsce)"`
3. Dodaj do `AVAILABLE_TOOLS` i do `tools_definition` (przez `make_tool`)

In [ ]:
# Ćwiczenie 2 — rozwiązanie

def get_population(city: str) -> str:
    dane = {
        "Warszawa": (1_860_000, 1), "Kraków": (800_000, 2), "Wrocław": (674_000, 3),
        "Gdańsk": (470_000, 6), "Poznań": (535_000, 5), "Łódź": (646_000, 4),
    }
    if city in dane:
        pop, rank = dane[city]
        return f"{city}: ~{pop:,} mieszkańców (#{rank} w Polsce)"
    return f"Brak danych o populacji dla: {city}"

class GetPopulationArgs(BaseModel):
    city: str = Field(..., description="Nazwa miasta, np. 'Kraków'")

# Usuwamy stary wpis (np. z nieuzupełnionej komórki powyżej)
tools_definition = [t for t in tools_definition if t["function"]["name"] != "get_population"]
tools_definition.append(
    make_tool("get_population", "Zwraca przybliżoną liczbę mieszkańców polskiego miasta.", GetPopulationArgs)
)
AVAILABLE_TOOLS["get_population"] = get_population

---
### 3. Ćwiczenie 3: Zbuduj narzędzie Wikipedia

**Zadanie:**
1. Napisz funkcję `search_wikipedia(query)` która zwraca tytuł, streszczenie i URL artykułu jako tekst
2. Dodaj definicję narzędzia do `tools_definition` + do `AVAILABLE_TOOLS`
3. Przetestuj pytaniem do LLM-a

In [ ]:
# Ćwiczenie 3 — rozwiązanie
import wikipedia
wikipedia.set_lang("pl")

def search_wikipedia(query: str) -> str:
    """Przeszukuje Wikipedię i zwraca tytuł + streszczenie artykułu."""
    try:
        results = wikipedia.search(query, results=3)
        if not results:
            return f"Nie znaleziono artykułów dla: {query}"

        try:
            page = wikipedia.page(results[0])
        except wikipedia.DisambiguationError as e:
            page = wikipedia.page(e.options[0])

        return f"{page.title}\n\n{page.summary[:500]}\n\nŹródło: {page.url}"

    except Exception as e:
        return f"Błąd wyszukiwania Wikipedia: {e}"

class SearchWikipediaArgs(BaseModel):
    query: str = Field(..., description="Zapytanie do Wikipedii, np. 'fotosynteza', 'Nikola Tesla'")

tools_definition = [t for t in tools_definition if t["function"]["name"] != "search_wikipedia"]
tools_definition.append(
    make_tool("search_wikipedia",
              "Przeszukuje Wikipedię i zwraca streszczenie artykułu. "
              "Użyj gdy pytanie dotyczy wiedzy ogólnej, historii, nauki, geografii.",
              SearchWikipediaArgs)
)
AVAILABLE_TOOLS["search_wikipedia"] = search_wikipedia

---
### 4. Ćwiczenie 4: FactCheck — zweryfikuj twierdzenie strukturalnie

Funkcja `verify_claim()` jest gotowa powyżej — brakuje jej tylko **modelu `FactCheck`**.

**Twoje zadanie:** Uzupełnij 5 pól Pydantic:
- `claim` — sprawdzane twierdzenie (str)
- `evidence` — znalezione dowody (str)
- `verdict` — werdykt: `Literal["prawda", "fałsz", "nie da się zweryfikować"]`
- `confidence` — pewność 0-1: `Field(..., ge=0, le=1)`
- `source` — skąd pochodzi dowód (str)

`instructor` wymusza, żeby LLM zwrócił obiekt dokładnie w formacie Twojego modelu.
Jeśli LLM pominie pole, instructor automatycznie ponawia zapytanie z feedbackiem!

In [ ]:
# Ćwiczenie 4 — rozwiązanie (tylko model — verify_claim jest gotowa powyżej)

class FactCheck(BaseModel):
    claim: str = Field(..., description="Sprawdzane twierdzenie")
    evidence: str = Field(..., description="Znalezione dowody potwierdzające lub obalające")
    verdict: Literal["prawda", "fałsz", "nie da się zweryfikować"] = Field(
        ..., description="Werdykt na podstawie dowodów"
    )
    confidence: float = Field(..., ge=0, le=1, description="Pewność werdyktu (0=zgaduję, 1=pewny)")
    source: str = Field(..., description="Skąd pochodzą dowody, np. 'baza prezydentów', 'Wikipedia'")